# ⚖️ Notebook 03 — MLE vs MAP: So sánh toàn diện

**Mục tiêu:**
- So sánh MLE và MAP trên nhiều tình huống
- Hiểu khi nào nên dùng cái nào
- Thấy mối liên hệ với regularization trong Machine Learning

**Điều kiện:** Đã xem notebook 01 và 02.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.mle import GaussianMLE, BernoulliMLE
from src.map import GaussianMAPEstimator, BernoulliMAPEstimator
from src.utils import make_gaussian, make_bernoulli, make_small_sample
from src.utils.plotting import plot_mle_vs_map_bernoulli, plot_mle_vs_map_gaussian

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('✅ Sẵn sàng!')

---
## Phần 1: Dashboard so sánh nhanh

Dùng các hàm plotting từ `src/utils/plotting.py` để so sánh nhanh.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bernoulli: n nhỏ ---
data_b = make_small_sample('bernoulli', seed=1)  # n=5, p thật=0.8
plot_mle_vs_map_bernoulli(data_b, alpha=2, beta=2, ax=axes[0])
axes[0].set_title(f'Bernoulli: n=5, data={list(data_b.astype(int))}\nPrior: Beta(2,2)', fontsize=10)

# --- Gaussian: n nhỏ ---
data_g = make_small_sample('gaussian', seed=1)  # n=5, μ thật=3
plot_mle_vs_map_gaussian(data_g, mu0=0.0, tau=1.5, sigma=1.0, ax=axes[1])
axes[1].set_title(f'Gaussian: n=5, x̄={np.mean(data_g):.2f}\nPrior: N(0, 1.5²)', fontsize=10)

plt.suptitle('MLE vs MAP — cả hai trường hợp với n nhỏ', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Phần 2: Thi đua — ai dự đoán đúng hơn khi n nhỏ?

Chạy 100 thí nghiệm: mỗi lần lấy n mẫu nhỏ, tính MLE và MAP, so sánh với giá trị thật.

In [ ]:
TRUE_P = 0.7
N_EXPERIMENTS = 200
ALPHA, BETA = 2, 2

n_values = [3, 5, 10, 20, 50]
mle_errors = {n: [] for n in n_values}
map_errors = {n: [] for n in n_values}

for seed in range(N_EXPERIMENTS):
    rng = np.random.default_rng(seed)
    for n in n_values:
        data = rng.binomial(1, TRUE_P, size=n).tolist()
        
        p_mle = np.mean(data)
        p_map = (ALPHA + sum(data) - 1) / (ALPHA + BETA + n - 2)
        
        mle_errors[n].append(abs(p_mle - TRUE_P))
        map_errors[n].append(abs(p_map - TRUE_P))

# Vẽ kết quả
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Mean Absolute Error ---
ax = axes[0]
mean_mle = [np.mean(mle_errors[n]) for n in n_values]
mean_map = [np.mean(map_errors[n]) for n in n_values]

x = np.arange(len(n_values))
w = 0.35
ax.bar(x - w/2, mean_mle, w, label='MLE', color='steelblue', alpha=0.85)
ax.bar(x + w/2, mean_map, w, label=f'MAP (Beta({ALPHA},{BETA}))', color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'n={n}' for n in n_values])
ax.set_ylabel('Mean Absolute Error |p_est - p_thật|')
ax.set_title(f'200 thí nghiệm — p thật={TRUE_P}\nMAP thắng khi n nhỏ!')
ax.legend()

# Thêm nhãn % improvement
for i, (m, mp) in enumerate(zip(mean_mle, mean_map)):
    improvement = (m - mp) / m * 100
    if improvement > 0:
        ax.text(i + w/2, mp + 0.003, f'+{improvement:.0f}%', 
               ha='center', fontsize=8, color='#c0392b', fontweight='bold')

# --- Boxplot phân phối lỗi ---
ax2 = axes[1]
n_focus = 5  # xem chi tiết tại n=5
bp = ax2.boxplot(
    [mle_errors[n_focus], map_errors[n_focus]],
    labels=[f'MLE\n(n={n_focus})', f'MAP Beta({ALPHA},{BETA})\n(n={n_focus})'],
    patch_artist=True,
    medianprops=dict(color='black', lw=2)
)
bp['boxes'][0].set_facecolor('#5dade2')
bp['boxes'][1].set_facecolor('#ec7063')

ax2.set_ylabel('|p_est - p_thật|')
ax2.set_title(f'Phân phối lỗi tại n={n_focus}\nMAP có phân phối hẹp hơn')

plt.tight_layout()
plt.show()

print('🏆 Kết quả:')
for n in n_values:
    m = np.mean(mle_errors[n])
    mp = np.mean(map_errors[n])
    winner = 'MAP 🏅' if mp < m else 'MLE 🏅'
    print(f'  n={n:3d}:  MLE error={m:.4f}  MAP error={mp:.4f}  → {winner}')

---
## Phần 3: MAP = MLE + Regularization

Đây là kết nối quan trọng nhất giữa Bayesian statistics và Machine Learning.

**MAP với Gaussian prior ↔ Ridge Regression (L2 regularization)**

```
MAP objective = log P(data|θ) + log P(θ)
              = log-likelihood - λ·||θ||²
```

Regularization không phải "hack" — nó có nền tảng Bayesian rõ ràng!

In [ ]:
# Minh họa: prior mạnh = regularization mạnh
np.random.seed(7)

# Data rất ít (n=4)
data_tiny = make_gaussian(n=4, mu=3.0, sigma=2.0, seed=7)
x_bar = np.mean(data_tiny)
sigma_known = 2.0
mu0 = 0.0

# Thay đổi tau (độ mạnh của prior = 1/lambda)
tau_values = [0.3, 0.7, 1.5, 5.0, 50.0]
mu_map_values = []

n = len(data_tiny)
for tau in tau_values:
    prec_lik = n / sigma_known**2
    prec_prior = 1 / tau**2
    mu_map = (prec_lik * x_bar + prec_prior * mu0) / (prec_lik + prec_prior)
    mu_map_values.append(mu_map)

lambda_equiv = [sigma_known**2 / tau**2 for tau in tau_values]  # λ tương đương

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- MAP vs tau ---
ax = axes[0]
ax.semilogx(tau_values, mu_map_values, 'o-', color='#e74c3c', lw=2, ms=8)
ax.axhline(x_bar, color='steelblue', lw=2, linestyle='--', label=f'MLE: μ={x_bar:.2f}')
ax.axhline(mu0, color='gray', lw=1.5, linestyle=':', label=f'Prior mean: μ₀={mu0}')
ax.axhline(3.0, color='green', lw=1.5, linestyle='-.', alpha=0.7, label='μ thật=3.0')

for tau, mu_map in zip(tau_values, mu_map_values):
    ax.annotate(f'τ={tau}', xy=(tau, mu_map), xytext=(0, 8),
               textcoords='offset points', ha='center', fontsize=8)

ax.set_xlabel('τ (prior std) — càng nhỏ = prior càng mạnh')
ax.set_ylabel('μ_MAP')
ax.set_title(f'MAP với n={n} — τ nhỏ = regularization mạnh')
ax.legend(fontsize=9)

# --- Bảng tương đương ---
ax2 = axes[1]
ax2.axis('off')
table_data = [
    ['Bayesian MAP', 'ML Regularization', 'Khi nào dùng?'],
    ['Gaussian prior\nN(0, τ²)', 'L2 / Ridge\nλ = σ²/τ²', 'Smooth weights'],
    ['Laplace prior\nLaplace(0,b)', 'L1 / Lasso\nλ = σ²/b', 'Sparse weights'],
    ['Uniform prior', 'Không regularize', 'Nhiều data'],
    ['Strong prior\n(τ nhỏ)', 'λ lớn\n(regularize mạnh)', 'Ít data'],
    ['Weak prior\n(τ lớn)', 'λ nhỏ\n(regularize nhẹ)', 'Nhiều data'],
]
table = ax2.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(9)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#ecf0f1')
ax2.set_title('MAP ↔ Regularization', fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

---
## Phần 4: Interactive — tự điều chỉnh và quan sát

Thay đổi các tham số bên dưới và chạy lại cell để quan sát tác động.

In [ ]:
# ============================================================
# 🎛️ THAY ĐỔI CÁC THAM SỐ NÀY VÀ CHẠY LẠI CELL
# ============================================================
N_SAMPLES = 8       # Cỡ mẫu (thử: 3, 8, 30, 100)
TRUE_P    = 0.7     # Xác suất thật (thử: 0.1, 0.5, 0.9)
ALPHA     = 2       # Prior α (thử: 1, 2, 5, 10)
BETA      = 2       # Prior β (thử: 1, 2, 5, 10)
SEED      = 42      # Seed để tái tạo (thử số khác)
# ============================================================

rng = np.random.default_rng(SEED)
data = rng.binomial(1, TRUE_P, size=N_SAMPLES).tolist()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_mle_vs_map_bernoulli(data, alpha=ALPHA, beta=BETA, ax=axes[0])

# Panel phải: thông tin tóm tắt
ax2 = axes[1]
ax2.axis('off')

k = sum(data)
n = len(data)
p_mle = k / n
p_map = (ALPHA + k - 1) / (ALPHA + BETA + n - 2)
prior_mean = ALPHA / (ALPHA + BETA)

info = [
    f'Data: {data}',
    f'k = {k} thành công / n = {n}',
    '',
    f'Prior: Beta({ALPHA}, {BETA})',
    f'  → Prior mean = {prior_mean:.3f}',
    '',
    f'p_MLE = k/n = {p_mle:.4f}',
    f'p_MAP = {p_map:.4f}',
    f'p thật = {TRUE_P}',
    '',
    f'|MLE - thật| = {abs(p_mle - TRUE_P):.4f}',
    f'|MAP - thật| = {abs(p_map - TRUE_P):.4f}',
    '',
    '🏅 ' + ('MAP' if abs(p_map - TRUE_P) < abs(p_mle - TRUE_P) else 'MLE') + ' gần đúng hơn lần này!'
]

for i, line in enumerate(info):
    weight = 'bold' if '🏅' in line or 'Prior' in line else 'normal'
    color = '#e74c3c' if '🏅' in line else 'black'
    ax2.text(0.05, 0.95 - i*0.07, line, transform=ax2.transAxes,
            fontsize=10, verticalalignment='top',
            fontweight=weight, color=color, fontfamily='monospace')

plt.suptitle('Interactive: thay đổi N_SAMPLES, ALPHA, BETA ở trên và chạy lại!',
             fontsize=11, style='italic')
plt.tight_layout()
plt.show()

---
## Phần 5: Khi nào dùng MLE, khi nào dùng MAP?

| Tình huống | Dùng gì? | Lý do |
|---|---|---|
| n lớn (>1000) | MLE | Prior không quan trọng |
| n nhỏ (<30) | MAP | Prior ổn định hóa |
| Biết rõ domain | MAP | Encode kiến thức vào prior |
| Không biết prior | MLE | Tránh bias sai |
| Cần uncertainty | Full Bayes | MAP chỉ cho điểm ước lượng |
| Production ML | MAP (regularize) | Tránh overfitting |

In [ ]:
print('=' * 60)
print('  CHECKLIST: MLE hay MAP?')
print('=' * 60)

questions = [
    ('n > 1000?',                      'MLE đủ tốt',               'Cần xem thêm'),
    ('Bạn có kiến thức về tham số?',   'Dùng MAP với prior đó',    'Prior uniform → MLE'),
    ('Kết quả không ổn định (n nhỏ)?', 'MAP với prior thích hợp',  'MLE ổn'),
    ('Cần quantify uncertainty?',       'Full Bayesian (MCMC/VI)',  'MAP hoặc MLE'),
]

for i, (q, yes, no) in enumerate(questions, 1):
    print(f'\n  {i}. {q}')
    print(f'     YES → {yes}')
    print(f'     NO  → {no}')

print('\n' + '=' * 60)
print('  BƯỚC TIẾP THEO:')
print('  → Làm bài tập trong exercises/beginner/')
print('  → Thử dùng src/ để giải bài toán của riêng bạn')
print('  → Tìm hiểu: PyMC, NumPyro để làm full Bayesian')
print('=' * 60)

---
## 🎓 Hoàn thành!

Bạn đã học xong 3 notebooks về MLE và MAP. Bước tiếp theo:

1. **Làm bài tập** trong `exercises/beginner/` (3 bài)
2. **Tạo Pull Request** để team lead review
3. **Khám phá thêm**: thử thay đổi tham số trong notebook này

Hỏi team lead nếu bạn bị stuck! 💪